# Predicting Californian house prices using Regression template

We are going to use the regression model introduced in DT401.5 to predict the median house value for [houses in California districts](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) and see how well linear regression can be used to make these predictions.

## The data

The data is taken from the 1990 U.S. census. Each row represents a census block group: the smallest geographical unit for which the U.S. Census Bureau publishes sample data. Each block group normally contains a population of 600 to 3,000 people.

Data Information:
- **MedHouseVal**:   The median house value, expressed in hundreds of thousands of dollars (\$100,000). This is the value we will look to predict.
- **MedInc**:        The median income, expressed in tens of thousands of dollars (\$10,000) in the block group
- **HouseAge**:      The median house age in the block group
- **AveRooms**:      The average number of rooms per household in the block group
- **AveBedrms**:     The average number of bedrooms per household in the block group
- **Population**:    The block group population
- **AveOccup**:      The average number of household members in the block group
- **Latitude**:      The block group latitude
- **Longitude**:     The block group longitude


A household is a group of people residing within a home. Since the average number of rooms and bedrooms in this dataset are provided per household, these columns may take surprisingly large values for block groups with few households and many empty houses, such as vacation resorts.

## What we need to consider for a regression

 For a regression we need:

- A dependent variable. This is the one we will predict. For a regression model this needs to be a continuous variable.
- One or more independent variables. These are the ones we are using to predict the dependent variable. For a regression all of these need to be numerical and at least some need to be continuous.

For a linear regression model, our hypothesis is that our dependent variable:

- Is influenced by each of the independent variables: the independent variable affects the dependent
- Has a relationship with each independent variable that is linear
- Has a similar distribution to each of the independent variables
- Is influenced more strongly by each independent variable than the influence they have on each other

## How to use this notebook

In this notebook all the code needed for the analysis has been written. You will be asked to reflect on the outputs of the code. Those tasks will be in one of two types of boxes:

<div class="alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the stages of the template with this dataset.
</div>

<div class="alert-success">
<b>Question boxes:</b> <br>
These have questions you should answer before progressing.
</div>

In both cases, if you wish to add your answers to this notebook as you progress, add an extra `markdown` cell to contain your answers. 

If you wish to save your answers at the end of the workshop, remember to download the notebook to your computer.

<a id='Model building tasks'></a>
## Model building steps
To build the model, we will:<b>

- [Import](#import) packages and load in some data 
- [Prepare](#prepare) the data so we can explore it
- [Explore](#explore) the data and make our testable hypothesis
- [Build](#build) the model 
- [Interpret](#interpret) the model results

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
You don't need to understand all the code here for now, just look for:
<ul>
<li> What we are trying to do with each code cell. How does it fit in our objective?
<li> What the outputs of each code cell tell us? Are we reading too much into the results of each code cell?
</ul>
</div>

<a id="import"></a>
## Import packages and read data
[Back to Contents](#Contents)

Let's start by importing the Python packages we will need:
- [**pandas**](https://pandas.pydata.org/): a tabular data manipulation package
- [**seaborn**](https://seaborn.pydata.org/): a data visualisation package
- [**numpy**](https://numpy.org/): a numerical analysis package
- [**statsmodels.api**](https://www.statsmodels.org/): a model building package

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
import seaborn as sns
import numpy as np
import statsmodels.api as sm

from sklearn.datasets import fetch_california_housing

sns.set_style("whitegrid")

We can now import the data using the Pandas function [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) and have a look at the first few rows using the method [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)

In [ ]:
independent_data, dependent_data = fetch_california_housing(return_X_y=True, as_frame=True)
data = independent_data.merge(dependent_data, left_index=True, right_index=True)
data.head()

<a id="prepare"></a>
### Prepare the data
[Back to Contents](#Contents)

For this notebook we are assuming the data has been prepared before being loaded in. However, it is always important to check that you have what you expect.

We can look at what kind of data our table contains using the [`info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) method. 

We should be asking ourselves:

- Does the data contain the columns I expect?
- Do the columns have the data type I would expect?
- Do any of the columns have missing values? Are these in any columns we intend to use? There cannot be any null values in the rows we plan to use in our model.

In [ ]:
data.info()

If we have any columns with null values that we want to use in our model, we will need to drop those pieces of data. Pandas gives us the [`dropna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) method to do this. If you only need to drop null values from one column you can use the `subset` parameter to pass a list of the columns you wish to be checked for null values.

This piece of code has been commented out, so it will not run. To use this, remove the `#` from the start of the line and replace `'variable name'` with the column that you wish to be checked for null values. You can examine more than one column at a time by listing all the columns in the square brackets.

<div class="alert-info">
<b>We don't need to drop any rows:</b> <br>
We can see from the output of the <tt>info</tt> method that there are no null values in any of the columns. We can move on to the next step.
</div>

In [ ]:
# data = data.dropna(subset=['variable name', ])

If one of the columns is only useful as a unique row identifier, we can use it as an index. We can set it using the [`set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) method. Replace `'index column name'` with the column that you wish to set as the index. If you need this, remove the `#` to uncomment the code.

<div class="alert-info">
<b>We don't need to set any rows as the index:</b> <br> <br>
All of the columns in this dataset contain useful information for our model and none of them are unique identifiers. We can move on to the next step.
</div>

In [ ]:
#data = data.set_index('ID column name')

---
<a id="explore"></a>
### Exploring the data
[Back to Contents](#Contents)

Now that we have checked our data is clean, we can explore it. A good starting point is to look at the descriptive statistics and check that they seem reasonable. We can do so using the [`describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) method.

<div class="alert-info">
<b>We should be asking ourselves:</b> <br>

- Does the data have unexpected outliers (looking at the max and min values) that might suggest a data quality issue?
- Is the spread of the data what you would expect?
</div>

In [ ]:
data.describe()

It is helpful to do these checks visually. Histograms will help us see the distribution of a variable and scatter plots will show us the relationship between variables. Seaborn allows us to plot these using the functions [`histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html) for histograms and [`scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) for scatter plots.

In [ ]:
sns.histplot(data=data, x='MedHouseVal');

In [ ]:
sns.scatterplot(data=data, x='MedInc', y='MedHouseVal');

It can be helpful to plot all the histograms and scatter plots in one go. Seaborn allows us to do this using a [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html).

In [ ]:
sns.pairplot(data);

The scatter plots help us to understand what kind of relationships there are between our variables and how strong they are. We can also quantify this using the Pandas method [`corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)

In [ ]:
data.corr(numeric_only=False)

We can visualise the correlation matrix using a heatmap, as we saw in the last workshop. We can use the Seaborn function [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) to do this.

In [ ]:
corr = data.corr(numeric_only=False)
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), 
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f");

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Explore the correlations between the different columns</p>
    <ul>
        <li> Which columns are most related to <tt>MedHouseVal</tt>? Keep in mind that a strong negative correlation is still a strong relationship.
        <li> To predict <tt>MedHouseVal</tt>, which columns do you think will be most useful? Keep in mind that to predict a value, the relationship needs to be strong and linear. To explain as much of the variation in <tt>MedHouseVal</tt> as possible, we need to use the variables that explain different parts of the variation, so it helps if they are not particularly related to each other.
    </ul>
</div>

### Confirm your hypothesis

Now that you have explored your data and seen the relationships in it, you can pick the independent variables you want to use to predict your dependent variable. Remember, for your independent variables to be useful in predicting the dependent variable, they need to:

- influence the dependent variable
- Have a linear relationship with the dependent variable
- Have a similar distribution to the dependent variable
- Influences more the dependent variable than any of the other chosen independent variables

---
<a id="build"></a>
## Build the model
[Back to Contents](#Contents)

Now we have our data in a form we can build our linear regression model.

Our predictor variables, the independent variables, are:

In [ ]:
independent_variable_names = ['MedInc', 'AveRooms', 'AveOccup', 'Latitude']

Our predicted variable, the dependent variable, is:

In [ ]:
dependent_variable_name = 'MedHouseVal'

We will use statsmodels Ordinary Least Squares [`ols`](https://www.statsmodels.org/dev/regression.html) function.

In [ ]:
independent_vars = data[independent_variable_names]
independent_vars = sm.add_constant(independent_vars)

data_model = sm.OLS(data[dependent_variable_name], independent_vars).fit()
data_model.summary()

---
<a id="interpret"></a>
## Interpret the model results
[Back to Contents](#Contents)

Now we have built the model we can look at the model quality and what it tells us.

- The **R squared** is a measure of how much of the variation in dependent variable is explained by the independent variables.
- The **Adj. R-squared** gives us an understanding of how much of the variation in dependent variable is explained by the independent variables. It is useful when there is more than one independent variable

The coefficients will give us the equation for our linear regression model line:

$$\hat{Y} = \text{const} + \text{coeff} \times X$$

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Look back at the model output:</p>
    <ul>
        <li> What are the coefficients for each of the independent variables and the constant? 
        <li> What are the units of these coefficients? 
        <li> How accurate are the estimated coefficients?
        <li> Do all of our independent variables contribute to the model?
        <li> What do these coefficients tell us about the relationship between the independent variables and the dependent variable?
        <li> How much of the variation in <tt>MedHouseVal</tt> is explained by the independent variables?
    </ul>
</div>